# 🔥 MODO BESTIA EN KAGGLE: Subagente 3 - UGC & Creadores (TikTok Shop / Reels)
### Plataforma: Kaggle Dual GPU (T4 x2) | Modelo: Meta-Llama-3.1-8B-Instruct | LoRA r=32

Este cuaderno entrena al especialista definitivo en **video de creadores en primera persona (UGC)**:
1. **Cámara Selfie 24mm Gran Angular:** Micro-jitter humano real en la mano que sostiene el teléfono.
2. **El 'Vicio de Pantalla':** Alterna la mirada entre la pantalla (2° abajo) y el lente en el punchline.
3. **Integridad del Producto:** Manos anatómicas de 5 dedos sosteniendo y rotando el empaque sin deformarse.
4. **Piel Real de Celular:** Cero filtro beauty, textura de poros auténtica y luz natural de ventana.

> **Importante:** Recuerda tener **Internet ON** y **GPU T4 x2** activados en el panel derecho de Kaggle.

## Paso 1: Instalación de Unsloth y Librerías de Optimización

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Paso 2: Cargar Meta-Llama-3.1-8B-Instruct en 4-Bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✅ Meta-Llama-3.1-8B cargado exitosamente en Kaggle.")

## Paso 3: Configurar LoRA de Alta Capacidad (Modo Bestia: r = 32, alpha = 64)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✅ Adaptadores LoRA r=32 configurados para el Especialista UGC.")

## Paso 4: Auto-Generar el Dataset UGC en Memoria (Cero archivos faltantes)

In [ ]:
import json
import os
from datasets import Dataset

UGC_EXPANDIDO = [
    ("Creadora recomendando serum facial de ácido hialurónico", "Ok, llevo dos semanas usando esto y mira mi piel… o sea, ¿qué? No lo puedo creer."),
    ("Joven mostrando app de productividad y finanzas personales", "Llevaba meses intentando ahorrar y esta tontería fue lo único que me funcionó de verdad."),
    ("Creador mostrando audífonos inalámbricos con cancelación de ruido", "Si trabajas en un café ruidoso necesitas esto ya mismo. Es literalmente magia."),
    ("Chica recomendando suplemento de magnesio para dormir mejor", "Mi nivel de energía cambió por completo en cinco días. No vuelvo a comprar otra marca."),
    ("Emprendedor mostrando teclado ergonómico mecánico para programar", "Mis dolores de muñeca desaparecieron después de una semana programando con esto."),
    ("Creadora de belleza mostrando labial de larga duración", "Comí pizza, tomé café todo el día y el labial sigue exactamente intacto."),
    ("Joven fitness probando creatina monohidratada sin sabor", "Se disuelve en cinco segundos sin grumos y no sabe a nada químico, recomendado total."),
    ("Creador tech mostrando cargador magnético ultrarrápido", "Carga el teléfono del diez al ochenta por ciento en veinte minutos, brutal para viajes.")
]

pares_ugc = []
for _ in range(5):
    for idea, speech in UGC_EXPANDIDO:
        chosen = (
            f"Maintain the exact creator, casual attire, room background, natural window light and product packaging from the first frame. The product never morphs or changes proportions.\n\n"
            f"CAMERA: Handheld smartphone selfie at arm's length, 24mm front-camera perspective, deep focus throughout. Continuous organic micro-jitter from hand holding phone, one small natural reframe when bringing product closer, brief focus hunting as product nears lens. No gimbal smoothness, no cuts.\n\n"
            f"SUBJECT: Talks fast and casually, mostly watching own screen just below the lens, but snaps direct eye contact to the lens on the punchline: \"{speech}\" Begins with skeptical frown, then eyebrows shoot up asymmetrically on emphasis. Tilts head, briefly glances down at product while rotating it toward camera, then back. Frequent natural blinks; short genuine laugh at the end. Free hand moves product naturally with anatomically correct fingers.\n\n"
            f"ENVIRONMENT: Lived-in interior with natural window daylight, casual morning atmosphere.\n\n"
            f"Audio: phone-microphone recording of natural upbeat voice, slight room acoustic reverb, faint background street noise, no music, no subtitles."
        )
        rejected = (
            f"Maintain creator. "
            f"CAMERA: Perfectly stabilized smooth gimbal tracking, cinematic f/1.4 blur. "
            f"SUBJECT: Actor stares rigidly into camera lens with porcelain airbrushed skin, product label blurs and transforms into alien text, formal teleprompter voice delivery, both hands waving awkwardly. "
            f"ENVIRONMENT: Studio 3-point lighting. Audio: studio voiceover."
        )
        pares_ugc.append({
            "prompt": f"Generate an Image-to-Video (I2V) prompt for Veo 3.1 based on this anchor: {idea}. Action: Fast casual smartphone selfie creator review",
            "chosen": chosen,
            "rejected": rejected
        })

archivo_dpo = "dataset_dpo_ugc.jsonl"
with open(archivo_dpo, "w", encoding="utf-8") as f:
    for item in pares_ugc:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

dataset = Dataset.from_list(pares_ugc)
print(f"✅ Dataset UGC auto-generado: {len(dataset)} pares DPO listos en /kaggle/working/.")

## Paso 5: Entrenamiento DPO en Kaggle (GPU T4 x2)

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    output_dir="./resultados_ugc",
    beta=0.15,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    warmup_ratio=0.1,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    save_strategy="no",
    seed=3407,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_length=2048,
    max_prompt_length=512,
)

gpu_mem_used = torch.cuda.memory_allocated() / 1024**3
print(f"📊 VRAM Inicial: {gpu_mem_used:.2f} GB")
print("🔥 Entrenando al Subagente UGC en Kaggle... (Duración aprox: 10-12 minutos)")
stats = dpo_trainer.train()
print("🎉 ¡Subagente UGC entrenado con éxito!")

## Paso 6: Guardar y Generar Enlace Directo de Descarga

In [ ]:
import shutil
from IPython.display import FileLink

out_dir = "lora_video_ugc_8b"
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)

zip_name = f"{out_dir}.zip"
shutil.make_archive(out_dir, 'zip', out_dir)
print(f"📦 Modelo empaquetado como: {zip_name}")

# Enlace HTML de descarga directa
print("\n👉 Haz clic en el siguiente enlace para descargar directamente a tu PC:")
display(FileLink(zip_name))

# Enlace temporal en la nube como respaldo (doble alternativa)
print('\n🌐 Subiendo a servidores temporales de respaldo (copia el link que aparezca abajo):')
!curl -T {zip_name} https://temp.sh/ || true
!curl -F "file=@{zip_name}" https://0x0.st || true
print('\n💡 Recuerda: También puedes descargar el zip desde el panel derecho de Kaggle -> Output -> lora_video_ugc_8b.zip -> Tres puntos (...) -> Download.')